In [9]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# ==========================================
# Student Name: Ahmad Hamizan bin Mohd Ali
# Student ID: BIS01086549
# ==========================================

def get_lazada_reviews(max_pages=5):
    """
    PURPOSE: This function connects to Trustpilot (a platform with user reviews)
    to scrape real customer reviews for Lazada Malaysia. It loops through a set
    number of pages and extracts the Name, Date, and Content.
    """
    all_reviews = []

    # Headers make our Python script look like a normal Google Chrome browser
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }

    # Loop exactly from page 1 to max_pages (limit to 5 pages as per requirements)
    for page in range(1, max_pages + 1):
        print(f"Scraping page {page}...")

        # The URL for Lazada Malaysia's reviews on Trustpilot
        url = f"https://www.trustpilot.com/review/www.lazada.com.my?page={page}"

        try:
            # Send the HTTP request and parse the HTML
            response = requests.get(url, headers=headers)
            soup = BeautifulSoup(response.content, 'html.parser')

            # Trustpilot wraps each individual user review in an <article> tag
            reviews = soup.find_all('article')

            # Stop if the page is empty
            if not reviews:
                print("No more reviews found on this page.")
                break

            # Go through each review one by one
            for review in reviews:

                # a. Extract Reviewer Name
                # Trustpilot stores names in an element with a specific data attribute
                name_elem = review.find(attrs={"data-consumer-name-typography": True})
                reviewer = name_elem.text.strip() if name_elem else "Anonymous"

                # b. Extract Review Date
                # Trustpilot uses standard <time> tags with a datetime attribute
                date_elem = review.find('time')
                if date_elem and 'datetime' in date_elem.attrs:
                    # Slice the string [:10] to keep only the YYYY-MM-DD format
                    review_date = date_elem['datetime'][:10]
                else:
                    review_date = "No Date"

                # c. Extract Review Content
                content_elem = review.find(attrs={"data-service-review-text-typography": True})
                review_content = content_elem.text.strip() if content_elem else "No written content"

                # Skip reviews where the user just left a star rating with no words
                if review_content == "No written content":
                    continue

                # Store the extracted information into our master list
                all_reviews.append({
                    "Reviewer Name": reviewer,
                    "Review Date": review_date,
                    "Review Content": review_content
                })

            # Pause for 1 second between pages to be polite to the server
            time.sleep(1)

        except Exception as e:
            print(f"Failed to scrape page {page}. Error: {e}")
            break

    return all_reviews

def main():
    """
    PURPOSE: Main function to run the scraping loop, convert the extracted data
    into a structured table, and save it as a CSV file.
    """
    # Call the scraper and strictly limit it to 5 pages
    review_data = get_lazada_reviews(max_pages=5)

    # Convert the list of dictionaries into a Pandas DataFrame
    df = pd.DataFrame(review_data)

    # Remove any duplicate reviews just in case
    df.drop_duplicates(inplace=True)

    # Save the data into a CSV file
    csv_filename = "lazada_malaysia_reviews.csv"
    df.to_csv(csv_filename, index=False)

    # Print a summary message
    print("\nScraping completed!")
    print(f"Total reviews scraped: {len(df)}")
    print(f"Data saved to: {csv_filename}")

# Run the script
if __name__ == '__main__':
    main()

Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...

Scraping completed!
Total reviews scraped: 97
Data saved to: lazada_malaysia_reviews.csv
